## 1. DC Power Flow Theory

### 1.1 AC to DC Linearization

The AC power flow equations are nonlinear and computationally expensive for real-time control. DC Power Flow (DCPF) provides a linearized approximation:

#### Full AC Power Flow Equations

Real and reactive power injections at bus $i$:

$$P_i = \sum_j |V_i||V_j|[G_{ij}\cos(\theta_{ij}) + B_{ij}\sin(\theta_{ij})]$$

$$Q_i = \sum_j |V_i||V_j|[G_{ij}\sin(\theta_{ij}) - B_{ij}\cos(\theta_{ij})]$$

where:
- $|V_i|, \theta_i$: Voltage magnitude and angle
- $G_{ij}, B_{ij}$: Conductance and susceptance
- $\theta_{ij} = \theta_i - \theta_j$: Angle difference

#### DC Power Flow Assumptions

For DC networks (or AC with specific assumptions):
1. **Slack bus reference**: $\theta_1 = 0$ (reference angle)
2. **Voltage magnitude**: $|V_i| = 1.0$ pu (constant)
3. **Small angle approximation**: $\sin(\theta_{ij}) \approx \theta_{ij}$, $\cos(\theta_{ij}) \approx 1$
4. **Negligible conductance**: $G_{ij} \ll B_{ij}$ (resistance << reactance)

#### Linearized DC Power Flow

Under these assumptions, the power flow simplifies to:

$$P_i = \sum_j B_{ij}(\theta_i - \theta_j)$$

In matrix form:

$$\mathbf{P} = \mathbf{B} \mathbf{\theta}$$

where $\mathbf{B}$ is the susceptance (admittance) matrix and $\mathbf{\theta}$ is the voltage angle vector.

**Solving for angles**:

$$\mathbf{\theta} = \mathbf{B}^{-1} \mathbf{P}$$

**Line flows** between buses $i$ and $j$:

$$P_{ij} = B_{ij}(\theta_i - \theta_j) = B_{ij} \Delta\theta_{ij}$$

### 1.2 Loss Calculation

Line losses (I²R loss) depend on the current squared:

$$P_{loss,ij} = I_{ij}^2 R_{ij} = \left(\frac{P_{ij}}{V}\right)^2 R_{ij} = \frac{P_{ij}^2}{V^2} R_{ij}$$

Total network losses:

$$P_{loss,total} = \sum_{(i,j) \in E} \frac{P_{ij}^2}{V^2} R_{ij}$$

In DC systems with unity voltage (V=1), this simplifies to:

$$P_{loss,total} = \sum_{(i,j) \in E} P_{ij}^2 R_{ij}$$

In [ ]:
# PART 1: DC POWER FLOW IMPLEMENTATION

import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Tuple, Dict, List
import sys
from pathlib import Path

# Configure visualization
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['font.size'] = 11

print("="*80)
print("1.1 DC POWER FLOW SOLUTION & LOSS CALCULATION")
print("="*80)

class DCPowerFlow:
    """DC Power Flow solver for microgrids"""
    
    def __init__(self, network: nx.Graph):
        """
        Initialize DC PF solver
        
        Parameters:
        -----------
        network : nx.Graph
            Network graph with edge attributes 'resistance' and 'reactance'
        """
        self.network = network
        self.num_nodes = network.number_of_nodes()
        self.num_edges = network.number_of_edges()
        
        # Build susceptance matrix
        self.B_matrix = self._build_susceptance_matrix()
    
    def _build_susceptance_matrix(self) -> np.ndarray:
        """Build susceptance (admittance) matrix B"""
        B = np.zeros((self.num_nodes, self.num_nodes))
        
        for u, v, data in self.network.edges(data=True):
            # Susceptance = 1 / reactance
            x = data.get('reactance', 0.1)  # Default reactance
            b = 1.0 / x if x != 0 else 0
            
            B[u, v] = -b
            B[v, u] = -b
            B[u, u] += b
            B[v, v] += b
        
        return B
    
    def solve(self, power_injections: np.ndarray) -> Dict:
        """
        Solve DC power flow
        
        Parameters:
        -----------
        power_injections : np.ndarray
            Power injections at each bus (pu)
        
        Returns:
        --------
        dict with angles, flows, losses
        """
        # Fix slack bus (node 0) at angle 0
        # Remove slack bus row/column for solution
        B_reduced = self.B_matrix[1:, 1:]
        p_reduced = power_injections[1:]
        
        try:
            # Solve: θ = B^{-1} P
            theta_reduced = np.linalg.solve(B_reduced, p_reduced)
            
            # Prepend slack bus angle
            theta = np.concatenate([[0], theta_reduced])
        except np.linalg.LinAlgError:
            print("Warning: Singular susceptance matrix. Using pseudoinverse.")
            B_pinv = np.linalg.pinv(self.B_matrix)
            theta = B_pinv @ power_injections
        
        # Calculate line flows and losses
        flows = {}
        losses = {}
        total_loss = 0.0
        
        for u, v, data in self.network.edges(data=True):
            x = data.get('reactance', 0.1)
            r = data.get('resistance', 0.01)
            b = 1.0 / x if x != 0 else 0
            
            # Flow: P_ij = B_ij * (θ_i - θ_j)
            p_flow = b * (theta[u] - theta[v])
            flows[(u, v)] = p_flow
            
            # Loss: P_loss = I² * R = (P/V)² * R ≈ P² * R (V=1)
            loss = (p_flow ** 2) * r
            losses[(u, v)] = loss
            total_loss += loss
        
        return {
            'angles': theta,
            'flows': flows,
            'losses': losses,
            'total_loss': total_loss,
        }

# Create example network
G = nx.Graph()
G.add_edges_from([
    (0, 1, {'reactance': 0.1, 'resistance': 0.01}),
    (0, 2, {'reactance': 0.15, 'resistance': 0.015}),
    (1, 2, {'reactance': 0.08, 'resistance': 0.008}),
    (1, 3, {'reactance': 0.12, 'resistance': 0.012}),
    (2, 3, {'reactance': 0.1, 'resistance': 0.01}),
])

# Add node attributes
for node in G.nodes():
    G.nodes[node]['name'] = f'Bus_{node}'

pf_solver = DCPowerFlow(G)

# Test case: Generation at bus 0, loads at other buses
power_inj = np.array([1.5, -0.5, -0.6, -0.4])  # Generation - Load
results = pf_solver.solve(power_inj)

print("\nDC Power Flow Solution:")
print(f"Slack bus (reference angle): θ₀ = 0 rad")
for i in range(1, len(results['angles'])):
    print(f"Bus {i}: θ_{i} = {results['angles'][i]:8.4f} rad ({np.degrees(results['angles'][i]):7.2f}°)")

print(f"\nLine Flows (pu):")
for (u, v), flow in results['flows'].items():
    print(f"  {u} → {v}: {flow:8.4f} pu")

print(f"\nLine Losses (pu):")
for (u, v), loss in results['losses'].items():
    print(f"  {u}-{v}: {loss:8.6f} pu")

print(f"\nTotal network loss: {results['total_loss']:.6f} pu ({results['total_loss']*100:.4f}%)")

## 2. Graph Representation & Feature Engineering

### 2.1 Node Features

Each node (bus) has features that characterize its operating point:

$$\mathbf{x}_i \in \mathbb{R}^{d_{node}}$$

**Standard node features** ($d_{node} = 6$):

| Feature | Symbol | Range | Units |
|---------|--------|-------|-------|
| Voltage magnitude | $V_i$ | [0.90, 1.10] | pu |
| Voltage angle | $\theta_i$ | [-π, π] | rad |
| Active power injection | $P_i$ | [-5, 5] | MW |
| Reactive power injection | $Q_i$ | [-2, 2] | MVAR |
| Frequency deviation | $\Delta f_i$ | [-0.5, 0.5] | Hz |
| Generator status | $G_i$ | {0, 1} | binary |

### 2.2 Edge Features

Each edge (line) connecting buses $i$ and $j$:

$$\mathbf{e}_{ij} \in \mathbb{R}^{d_{edge}}$$

**Standard edge features** ($d_{edge} = 5$):

| Feature | Symbol | Range | Units |
|---------|--------|-------|-------|
| Resistance | $R_{ij}$ | [0.001, 0.1] | pu |
| Reactance | $X_{ij}$ | [0.01, 0.5] | pu |
| Rated capacity | $S_{ij}^{max}$ | [10, 500] | MVA |
| Current flow | $I_{ij}$ | [0, 1] | pu |
| Line status | $s_{ij}$ | {0, 1} | binary |

### 2.3 Graph Construction

The microgrid is represented as a property graph:

$$G = (V, E, X_v, X_e, A)$$

where:
- $V$: Set of nodes (buses)
- $E$: Set of edges (transmission lines)
- $X_v \in \mathbb{R}^{|V| \times d_{node}}$: Node feature matrix
- $X_e \in \mathbb{R}^{|E| \times d_{edge}}$: Edge feature matrix
- $A \in \{0,1\}^{|V| \times |V|}$: Adjacency matrix

In [ ]:
# PART 2: GRAPH REPRESENTATION & FEATURE ENGINEERING

print("\n" + "="*80)
print("2.1 GRAPH REPRESENTATION FOR GNN INPUT")
print("="*80)

class GraphRepresentation:
    """Convert power system to GNN-compatible graph representation"""
    
    def __init__(self, network: nx.Graph):
        self.network = network
        self.node_features = {}
        self.edge_features = {}
    
    def compute_node_features(self, voltages: np.ndarray, angles: np.ndarray, 
                              power_inj: np.ndarray, freq_dev: np.ndarray) -> np.ndarray:
        """
        Compute normalized node features
        
        Parameters:
        -----------
        voltages : np.ndarray, shape (n_nodes,)
            Voltage magnitude (pu)
        angles : np.ndarray, shape (n_nodes,)
            Voltage angle (rad)
        power_inj : np.ndarray, shape (n_nodes,)
            Active power injection (MW)
        freq_dev : np.ndarray, shape (n_nodes,)
            Frequency deviation (Hz)
        
        Returns:
        --------
        np.ndarray, shape (n_nodes, 6)
            Normalized node features
        """
        n_nodes = self.network.number_of_nodes()
        
        # Initialize feature matrix
        X_node = np.zeros((n_nodes, 6))
        
        # Normalize and store features
        X_node[:, 0] = (voltages - 1.0) / 0.1  # Normalize voltage to ~[-1, 1]
        X_node[:, 1] = angles / np.pi  # Normalize angle to ~[-1, 1]
        X_node[:, 2] = power_inj / 5.0  # Normalize power to ~[-1, 1]
        X_node[:, 3] = freq_dev / 0.5  # Normalize frequency to ~[-1, 1]
        X_node[:, 4] = (voltages >= 0.95).astype(float)  # Voltage OK flag
        X_node[:, 5] = (power_inj > 0).astype(float)  # Generator flag
        
        self.node_features = X_node
        return X_node
    
    def compute_edge_features(self, line_flows: Dict) -> np.ndarray:
        """
        Compute normalized edge features
        
        Parameters:
        -----------
        line_flows : dict
            Line flows from DC power flow solution
        
        Returns:
        --------
        np.ndarray, shape (n_edges, 5)
            Normalized edge features
        """
        edges = list(self.network.edges())
        n_edges = len(edges)
        
        # Initialize feature matrix
        X_edge = np.zeros((n_edges, 5))
        
        for idx, (u, v) in enumerate(edges):
            edge_data = self.network[u][v]
            
            # Features
            r = edge_data.get('resistance', 0.01)
            x = edge_data.get('reactance', 0.1)
            s_max = edge_data.get('capacity', 100.0)  # MVA
            flow = line_flows.get((u, v), 0.0)
            status = edge_data.get('status', 1)  # 1 = connected
            
            # Normalize
            X_edge[idx, 0] = r / 0.1  # Resistance
            X_edge[idx, 1] = x / 0.5  # Reactance
            X_edge[idx, 2] = s_max / 500.0  # Capacity
            X_edge[idx, 3] = flow / 1.0  # Flow (normalized to 1 pu)
            X_edge[idx, 4] = status  # Binary status
        
        self.edge_features = X_edge
        return X_edge
    
    def get_adjacency_matrix(self) -> np.ndarray:
        """Return adjacency matrix"""
        return nx.to_numpy_array(self.network)
    
    def get_edge_index(self) -> Tuple[np.ndarray, np.ndarray]:
        """Return edge index (PyTorch geometric format)"""
        edges = np.array(list(self.network.edges())).T
        return edges[0], edges[1]

# Compute graph representation
graph_repr = GraphRepresentation(G)

# Generate sample state
voltages = np.array([1.00, 0.98, 0.99, 0.97])  # Voltage at each bus
angles = np.array([0.0, -0.05, -0.08, -0.12])  # Voltage angle at each bus
power_inj = np.array([1.5, -0.5, -0.6, -0.4])  # Power injection
freq_dev = np.array([0.0, -0.01, -0.02, -0.01])  # Frequency deviation

X_node = graph_repr.compute_node_features(voltages, angles, power_inj, freq_dev)
X_edge = graph_repr.compute_edge_features(results['flows'])
A = graph_repr.get_adjacency_matrix()

print("\nNode Feature Matrix X_node (shape: 4×6):")
print("Columns: [ΔV, θ, P, Δf, V_OK, Gen_flag]")
for i, features in enumerate(X_node):
    print(f"  Bus {i}: {features}")

print("\nEdge Feature Matrix X_edge (shape: 5×5):")
print("Columns: [R, X, S_max, I, Status]")
edges = list(G.edges())
for idx, (u, v) in enumerate(edges):
    print(f"  Edge {u}-{v}: {X_edge[idx]}")

print("\nAdjacency Matrix A (shape: 4×4):")
print(A)

print(f"\nGraph Statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Node feature dimension: {X_node.shape[1]}")
print(f"  Edge feature dimension: {X_edge.shape[1]}")
print(f"  Total parameters: {X_node.size + X_edge.size + A.size}")

## 3. Graph Neural Network Architecture

### 3.1 Graph Convolutional Networks (GCN)

Graph Convolutional Networks perform message passing to aggregate information from neighboring nodes:

#### Forward Pass

The update rule for a GCN layer is:

$$\mathbf{X}^{(l+1)} = \sigma\left(\tilde{\mathbf{D}}^{-1/2} \tilde{\mathbf{A}} \tilde{\mathbf{D}}^{-1/2} \mathbf{X}^{(l)} \mathbf{W}^{(l)}\right)$$

where:
- $\tilde{\mathbf{A}} = \mathbf{A} + \mathbf{I}$: Adjacency with self-loops
- $\tilde{\mathbf{D}}$: Degree matrix of $\tilde{\mathbf{A}}$
- $\mathbf{W}^{(l)}$: Learnable weight matrix
- $\sigma(\cdot)$: Activation function (ReLU)

**Intuition**: Each node aggregates features from its neighbors (via $\tilde{\mathbf{A}}$), applies learned transformations ($\mathbf{W}^{(l)}$), and applies nonlinearity ($\sigma$).

#### Multi-Layer Stack

Stacking $L$ GCN layers increases the receptive field (neighborhood horizon):

$$\text{Receptive Field} = L \text{ hops}$$

A 4-layer GCN can aggregate information from 4 hops away (entire network in small grids).

### 3.2 Multi-Task Output Heads

After GCN encoding, separate task-specific heads predict different objectives:

**Task 1: Voltage Violations** (node-level)
$$\hat{v}_i = \text{sigmoid}(\mathbf{W}_{v1} \mathbf{h}_i^{(L)} + b_{v1})$$

**Task 2: Thermal Violations** (edge-level)
$$\hat{t}_{ij} = \text{sigmoid}(\mathbf{W}_{t1} (\mathbf{h}_i^{(L)} \oplus \mathbf{h}_j^{(L)}) + b_{t1})$$

where $\oplus$ denotes concatenation

**Task 3: Total Loss** (graph-level)
$$\hat{l}_{loss} = \text{ReLU}(\mathbf{W}_{l1} \text{mean}(\mathbf{H}^{(L)}) + b_{l1})$$

**Task 4: Stability Margin** (graph-level)
$$\hat{m}_{stab} = \text{ReLU}(\mathbf{W}_{m1} \text{max}(\mathbf{H}^{(L)}) + b_{m1})$$

In [ ]:
# PART 3: GRAPH NEURAL NETWORK ARCHITECTURE

print("\n" + "="*80)
print("3.1 MULTI-TASK GRAPH CONVOLUTIONAL NETWORK")
print("="*80)

# Display GNN architecture
arch_diagram = f"""
╔════════════════════════════════════════════════════════════════════════════╗
║                    MULTI-TASK GNN ARCHITECTURE                            ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  INPUT LAYER                                                              ║
║  ├─ Node features: X_node ∈ ℝ^(n × 6)                                     ║
║  │   [V, θ, P, Δf, V_OK, Gen]                                             ║
║  ├─ Edge features: X_edge ∈ ℝ^(m × 5)                                     ║
║  │   [R, X, S_max, I, Status]                                             ║
║  └─ Adjacency: A ∈ {{0,1}}^(n × n)                                          ║
║                                                                            ║
║  GRAPH CONVOLUTIONAL BLOCKS (4 layers)                                    ║
║  ├─ Layer 1: GCNConv(6 → 64)  + BatchNorm + ReLU + Dropout(0.2)          ║
║  │   Message: h_i ← Σ_j (D^{-1/2} A D^{-1/2})_{ij} h_j W                  ║
║  │                                                                        ║
║  ├─ Layer 2: GCNConv(64 → 128) + BatchNorm + ReLU + Dropout(0.2)         ║
║  │   Receptive field: 2-hop neighborhood                                  ║
║  │                                                                        ║
║  ├─ Layer 3: GCNConv(128 → 128) + BatchNorm + ReLU + Dropout(0.2)        ║
║  │   Receptive field: 3-hop neighborhood                                  ║
║  │                                                                        ║
║  └─ Layer 4: GCNConv(128 → 64) + BatchNorm + ReLU                         ║
║      Receptive field: 4-hop (entire network)                             ║
║                                                                            ║
║  MULTI-TASK OUTPUT HEADS                                                 ║
║  ├─ Head 1: Voltage Violations (node-level)                               ║
║  │  Dense(64 → 32 → 1) sigmoid                                            ║
║  │  Output: ŷ_v ∈ [0, 1] per node                                         ║
║  │                                                                        ║
║  ├─ Head 2: Thermal Violations (edge-level)                               ║
║  │  Dense(128 → 64 → 1) sigmoid                                           ║
║  │  Output: ŷ_t ∈ [0, 1] per edge                                         ║
║  │                                                                        ║
║  ├─ Head 3: Power Loss (graph-level)                                      ║
║  │  MeanPooling → Dense(64 → 32 → 1) ReLU                                ║
║  │  Output: ŷ_l ∈ [0, ∞)                                                  ║
║  │                                                                        ║
║  └─ Head 4: Stability Margin (graph-level)                                ║
║     MaxPooling → Dense(64 → 32 → 1) ReLU                                 ║
║     Output: ŷ_m ∈ [0, ∞)                                                  ║
║                                                                            ║
║  LOSS FUNCTION (Weighted Multi-Task)                                      ║
║  L_total = 0.30·L₁ + 0.30·L₂ + 0.20·L₃ + 0.20·L₄                          ║
║                                                                            ║
║  PARAMETERS                                                               ║
║  ├─ GCN weights: 4 layers × (input_dim → output_dim)                      ║
║  ├─ Batch norm: 4 × 64 parameters                                         ║
║  └─ Output heads: ~5000 parameters                                        ║
║  Total: ~50,000 trainable parameters                                      ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(arch_diagram)

# Simulate GNN forward pass
class SimpleGNNBlock:
    """Simplified GCN layer for demonstration"""
    
    def __init__(self, in_dim: int, out_dim: int, name: str = ""):
        self.in_dim = in_dim
        self.out_dim = out_dim
        self.name = name
        # Learnable parameters
        self.W = np.random.randn(in_dim, out_dim) * np.sqrt(2.0 / in_dim)
        self.b = np.zeros(out_dim)
    
    def forward(self, X: np.ndarray, A: np.ndarray) -> np.ndarray:
        """
        GCN forward pass: X' = σ(D^{-1/2} A D^{-1/2} X W)
        """
        # Add self-loops
        A_tilde = A + np.eye(A.shape[0])
        
        # Compute degree matrix
        D = np.diag(np.sum(A_tilde, axis=1))
        D_inv_sqrt = np.diag(1.0 / np.sqrt(np.diag(D) + 1e-8))
        
        # Normalize: D^{-1/2} A D^{-1/2}
        A_norm = D_inv_sqrt @ A_tilde @ D_inv_sqrt
        
        # Message aggregation and transformation
        Z = A_norm @ X @ self.W + self.b
        
        # ReLU activation
        H = np.maximum(Z, 0)
        
        return H

# Build GNN stack
gnn_blocks = [
    SimpleGNNBlock(6, 64, "GCN_1"),
    SimpleGNNBlock(64, 128, "GCN_2"),
    SimpleGNNBlock(128, 128, "GCN_3"),
    SimpleGNNBlock(128, 64, "GCN_4"),
]

# Forward pass
H = X_node.copy()
print(f"\nGNN Forward Pass:")
print(f"Input shape: {H.shape}")

for block in gnn_blocks:
    H = block.forward(H, A)
    print(f"{block.name} output: {H.shape}")

print(f"\nFinal node embeddings: {H.shape}")
print(f"Average embedding magnitude: {np.mean(np.abs(H)):.4f}")
print(f"Max embedding: {np.max(H):.4f}")

## 4. Constraint Checking & Physics-Informed Losses

### 4.1 Constraint Violations

The GNN must learn to identify physical constraint violations:

**Voltage limits**:
$$V_i \in [V_{min}, V_{max}] = [0.95, 1.05] \text{ pu}$$

**Current/Thermal limits**:
$$I_{ij} \leq I_{ij}^{max}, \quad \Rightarrow \quad |P_{ij}| \leq S_{ij}^{max}$$

**Frequency stability**:
$$|\Delta f_i| \leq 0.5 \text{ Hz}$$

### 4.2 Physics-Informed Loss

We augment the standard supervised loss with physical penalty terms:

$$L_{total} = L_{supervised} + \lambda_{phys} L_{physics}$$

where the physics loss penalizes violations:

$$L_{physics} = \sum_i \max(0, |V_i| - 1.05)^2 + \max(0, 0.95 - |V_i|)^2 + \sum_j \max(0, |I_j| - I_j^{max})^2$$

In [ ]:
# PART 4: CONSTRAINT CHECKING & PHYSICS-INFORMED LOSSES

print("\n" + "="*80)
print("4.1 CONSTRAINT VIOLATION DETECTION & PHYSICS-INFORMED LOSS")
print("="*80)

class ConstraintChecker:
    """Check power system constraint violations"""
    
    def __init__(self, network: nx.Graph):
        self.network = network
        self.v_min, self.v_max = 0.95, 1.05
        self.f_max = 0.5
    
    def check_voltage_violations(self, voltages: np.ndarray) -> List[Tuple[int, float]]:
        """Return list of (bus_id, violation_amount)"""
        violations = []
        for i, v in enumerate(voltages):
            if v < self.v_min:
                violations.append((i, self.v_min - v))
            elif v > self.v_max:
                violations.append((i, v - self.v_max))
        return violations
    
    def check_thermal_violations(self, flows: Dict, capacities: np.ndarray = None) -> List[Tuple[Tuple[int,int], float]]:
        """Return list of ((from, to), violation_amount)"""
        violations = []
        for (u, v), flow in flows.items():
            # Default capacity 1 pu
            s_max = capacities.get((u, v), 1.0) if capacities else 1.0
            if abs(flow) > s_max:
                violations.append(((u, v), abs(flow) - s_max))
        return violations
    
    def check_frequency_violations(self, freq_dev: np.ndarray) -> List[Tuple[int, float]]:
        """Return list of (bus_id, violation_amount)"""
        violations = []
        for i, f in enumerate(freq_dev):
            if abs(f) > self.f_max:
                violations.append((i, abs(f) - self.f_max))
        return violations
    
    def compute_physics_loss(self, voltages: np.ndarray, freq_dev: np.ndarray, 
                            flows: Dict, lambda_phys: float = 1.0) -> float:
        """Compute physics penalty loss"""
        loss = 0.0
        
        # Voltage limit penalties
        for v in voltages:
            if v < self.v_min:
                loss += (self.v_min - v) ** 2
            elif v > self.v_max:
                loss += (v - self.v_max) ** 2
        
        # Frequency limit penalties
        for f in freq_dev:
            if abs(f) > self.f_max:
                loss += (abs(f) - self.f_max) ** 2
        
        # Thermal limit penalties (assume 1.0 pu capacity)
        for (u, v), flow in flows.items():
            if abs(flow) > 1.0:
                loss += (abs(flow) - 1.0) ** 2
        
        return lambda_phys * loss

# Test constraint checking
constraint_checker = ConstraintChecker(G)

# Test case with some violations
test_voltages = np.array([1.02, 0.93, 1.08, 0.96])  # Bus 1 low, Bus 2 high
test_freq = np.array([0.0, -0.1, 0.2, -0.05])  # All within limits
test_flows = {(0, 1): 1.2, (0, 2): 0.8, (1, 2): 0.5, (1, 3): 1.5, (2, 3): 0.3}  # Some overloads

volt_viols = constraint_checker.check_voltage_violations(test_voltages)
therm_viols = constraint_checker.check_thermal_violations(test_flows)
freq_viols = constraint_checker.check_frequency_violations(test_freq)
phys_loss = constraint_checker.compute_physics_loss(test_voltages, test_freq, test_flows)

print("\nConstraint Violation Analysis:")
print(f"\nVoltage violations (limit: [0.95, 1.05] pu):")
if volt_viols:
    for bus_id, violation in volt_viols:
        print(f"  Bus {bus_id}: {violation*100:.2f}% (V = {test_voltages[bus_id]:.3f} pu)")
else:
    print("  None")

print(f"\nThermal violations (limit: 1.0 pu):")
if therm_viols:
    for (u, v), violation in therm_viols:
        print(f"  Line {u}-{v}: {violation*100:.2f}% (I = {test_flows[(u, v)]:.3f} pu)")
else:
    print("  None")

print(f"\nFrequency violations (limit: ±0.5 Hz):")
if freq_viols:
    for bus_id, violation in freq_viols:
        print(f"  Bus {bus_id}: {violation*100:.2f}% (Δf = {test_freq[bus_id]:.3f} Hz)")
else:
    print("  None")

print(f"\nPhysics-Informed Loss Calculation:")
print(f"  L_physics = {phys_loss:.6f}")
print(f"  If L_supervised = 0.5, then:")
print(f"  L_total = L_sup + λ·L_phys = 0.5 + 1.0 × {phys_loss:.6f} = {0.5 + phys_loss:.6f}")

## 5. Conclusion: Complete GNN Framework

### Summary of Power Flow & GNN Design

This notebook has established the complete technical foundation:

1. **DC Power Flow**: Linearized AC-to-DC formulation with loss calculation
2. **Graph Features**: Node (6D) and edge (5D) feature engineering
3. **GNN Architecture**: 4-layer GCN with multi-task output heads
4. **Constraint Handling**: Physics-informed loss with penalty terms
5. **Multi-Task Learning**: Simultaneous prediction of 4 system objectives

### Key Insights
- **Receptive field grows exponentially**: 4 GCN layers → information from 4 hops away
- **Multi-task learning enables transfer**: Shared hidden layers learn common representations
- **Physics-informed loss ensures feasibility**: Constraint violations explicitly penalized
- **Graph structure preserves topology**: Networks of different sizes/topologies trainable together

### Next Steps
- **Part 4**: GNN training pipeline, hyperparameter optimization
- **Part 5**: Benchmark comparisons, real-world case studies

In [ ]:
print("\n" + "="*80)
print("GNN FRAMEWORK: COMPLETE ARCHITECTURE SUMMARY")
print("="*80)

summary = f"""
╔════════════════════════════════════════════════════════════════════════════╗
║               DS-PAH-GNN TECHNICAL ARCHITECTURE SUMMARY                    ║
╠════════════════════════════════════════════════════════════════════════════╣
║                                                                            ║
║  1. POWER FLOW LAYER                                                      ║
║     ├─ DC Power Flow: P = B·θ (linearized)                                ║
║     ├─ Loss calculation: P_loss = I²·R = (P/V)²·R                         ║
║     └─ Constraint checking: Voltage [0.95, 1.05], Thermal, Frequency      ║
║                                                                            ║
║  2. FEATURE ENGINEERING LAYER                                             ║
║     ├─ Node features (6D): V, θ, P, Δf, V_OK, Gen                        ║
║     ├─ Edge features (5D): R, X, S_max, I, Status                         ║
║     └─ Adjacency matrix: 0/1 connectivity                                 ║
║                                                                            ║
║  3. GNN ENCODER LAYER                                                     ║
║     ├─ 4 stacked GCN blocks: 6 → 64 → 128 → 128 → 64                      ║
║     ├─ Message passing: Aggregate from 4-hop neighborhood                 ║
║     ├─ Non-linearity: ReLU + Batch Normalization                          ║
║     └─ Regularization: Dropout (p=0.2)                                    ║
║                                                                            ║
║  4. MULTI-TASK DECODER LAYER                                              ║
║     ├─ Voltage violations: Node-level sigmoid                             ║
║     ├─ Thermal violations: Edge-level sigmoid                             ║
║     ├─ Power loss: Graph-level ReLU (sum pooling)                         ║
║     └─ Stability margin: Graph-level ReLU (max pooling)                   ║
║                                                                            ║
║  5. LOSS FUNCTION (Weighted Multi-Task)                                   ║
║     ├─ Supervised: L_sup = 0.30·BCE(V) + 0.30·BCE(T) + ...                ║
║     ├─ Physics-informed: L_phys = penalties for constraint violations      ║
║     └─ Total: L = L_sup + λ·L_phys (λ = 1.0)                              ║
║                                                                            ║
║  6. TRAINING CONFIGURATION                                                ║
║     ├─ Optimizer: Adam (lr=0.001, β₁=0.9, β₂=0.999)                       ║
║     ├─ Scheduler: StepLR (decay by 0.1 every 50 epochs)                   ║
║     ├─ Batch size: 32 (number of graphs per batch)                        ║
║     ├─ Epochs: 200-500 (depending on dataset size)                        ║
║     └─ Early stopping: patience=30 epochs on validation loss               ║
║                                                                            ║
║  7. DATASET GENERATION (FROM PART 2)                                      ║
║     ├─ Scenarios: 6,480 unique (topology × size × EV × solar × time)      ║
║     ├─ Timesteps: 100 per scenario                                        ║
║     ├─ Total samples: 432,000                                              ║
║     ├─ Train/Val/Test: 70% / 15% / 15%                                    ║
║     └─ Average graph size: 10 nodes, 15 edges                              ║
║                                                                            ║
║  8. EXPECTED PERFORMANCE                                                  ║
║     ├─ Voltage prediction: RMSE < 0.02 pu                                 ║
║     ├─ Loss prediction: MAPE < 10%                                        ║
║     ├─ Violation detection: F1 > 0.90                                      ║
║     └─ Inference time: ~5 ms per graph (GPU)                              ║
║                                                                            ║
╚════════════════════════════════════════════════════════════════════════════╝
"""

print(summary)
print("✓ GNN architecture complete and physics-informed")
print("✓ Ready for training and validation (Part 4)")